In [1]:
print("hi")

hi


Generate data

In [2]:
import numpy as np
import math

In [7]:
L, d_k, d_v = 4, 10, 10
q = np.random.randn(L, d_k)
k = np.random.randn(L, d_k)
v = np.random.randn(L, d_v)

In [9]:
print("Q\n", q)
print("K\n", k)
print("V\n", v)

Q
 [[-0.62216175 -0.87703221 -0.8588049   0.03792949 -0.20157045  1.52719172
   0.06464875 -1.03676197 -0.12062276 -1.21094341]
 [-1.01937969 -0.41035475 -0.19833812 -0.66828901  1.100473   -0.58715359
   0.56277128  0.82087674  1.15482673 -0.26369845]
 [ 0.95904511 -0.51248354  0.38496119 -0.13471253  1.18755381 -1.26213277
  -0.23695832 -0.58007255  0.08457898 -0.67136831]
 [-0.4952777   0.2332489   0.98893229 -0.93413198  1.41187294 -0.01222578
  -0.47614516 -0.50825264 -0.42175741  0.37247078]]
K
 [[-1.01024684  0.84562912  1.30285974 -0.22746246 -0.07658451  0.37897226
  -0.64012633  1.40195369 -0.1959627   0.9703588 ]
 [ 0.15327607 -1.67591322  1.29513144  1.20657875 -0.80070956  0.74721339
   0.46489243  0.19798146  0.68631049 -0.32847494]
 [ 0.92051831  0.1253748  -0.57490041  0.471459    0.08155867 -1.30435789
   1.13935736  0.91164766  0.68222847  1.49772013]
 [ 1.15367826  0.72312258  0.22482272  1.93959019 -0.93613158 -0.26276539
   1.1286182   0.13041126 -1.82527202  0.569

## Self Attention

$$
\text{self attention} = softmax\bigg(\frac{Q.K^T}{\sqrt{d_k}}+M\bigg)
$$

$$
\text{new V} = \text{self attention}.V
$$ 

Why need to divide by squroot(d_k)? <br>
Answer is to minimize the variance

In [13]:
np.matmul(q, k.T) 

array([[-3.29272411,  1.75027949, -4.94695411, -2.21560393],
       [ 0.57802836, -0.54829858,  1.44721667, -5.20527805],
       [-2.7689064 , -0.49848615,  0.5302468 , -1.09866472],
       [ 2.12207939, -2.1865458 , -2.04023864, -2.93248938]])

In [14]:
q.var(), k.var(), v.var(), np.matmul(q, k.T).var()

(np.float64(0.558129620656319),
 np.float64(0.7801816489932827),
 np.float64(1.030890869137362),
 np.float64(4.761762024339901))

See the variance is very high then q,k,v

In [15]:
scale = np.matmul(q, k.T) / math.sqrt(d_k)
q.var(), k.var(), v.var(), scale.var()

(np.float64(0.558129620656319),
 np.float64(0.7801816489932827),
 np.float64(1.030890869137362),
 np.float64(0.47617620243399006))

In [27]:
scale

array([[-1.04125079,  0.55348697, -1.56436425, -0.70063548],
       [ 0.18278862, -0.17338724,  0.45765009, -1.64605345],
       [-0.87560509, -0.15763516,  0.16767876, -0.34742829],
       [ 0.67106042, -0.69144649, -0.64518011, -0.92733456]])

## Masking

- This is to ensure words don't get context from words generated in the future. 
- Not required in the encoders, but required in the decoders.

In [17]:
mask = np.tril(np.ones( (L, L) ))
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [19]:
mask[mask == 0] = -np.inf
mask[mask == 1] = 0

In [20]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [22]:

scale + mask

array([[-1.04125079,        -inf,        -inf,        -inf],
       [ 0.18278862, -0.17338724,        -inf,        -inf],
       [-0.87560509, -0.15763516,  0.16767876,        -inf],
       [ 0.67106042, -0.69144649, -0.64518011, -0.92733456]])

## Softmax

$$
\text{softmax} = \frac{e^{x_i}}{\sum_j e^x_j}
$$

In [24]:
def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)
   

In [25]:

attention = softmax(scale + mask)

In [26]:
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.5881144 , 0.4118856 , 0.        , 0.        ],
       [0.16981418, 0.34816437, 0.48202145, 0.        ],
       [0.57924659, 0.14829764, 0.15532004, 0.11713572]])

In [28]:
new_v = np.matmul(attention, v)
new_v

array([[ 0.54526819,  0.12160048, -2.94736517,  0.42157008,  0.09842667,
         0.84614687, -0.10225613,  0.8286375 ,  0.94980427,  0.59018849],
       [ 0.67107329, -0.10776148, -1.70257461,  0.45734894,  0.75511262,
         0.39642414,  0.01550463,  0.32450663,  1.01469047,  0.45370622],
       [ 0.36929919, -0.83654625, -1.67326999, -0.29721843,  0.42433858,
         1.14977469,  0.60401288, -0.12535825,  0.37270174, -0.15468236],
       [ 0.53963383, -0.2522171 , -2.19263889,  0.1658323 ,  0.35462908,
         0.9936331 ,  0.03039358,  0.41455174,  0.69379944,  0.38803019]])

In [29]:

v

array([[ 0.54526819,  0.12160048, -2.94736517,  0.42157008,  0.09842667,
         0.84614687, -0.10225613,  0.8286375 ,  0.94980427,  0.59018849],
       [ 0.85070519, -0.43525793,  0.07481032,  0.50843609,  1.69276731,
        -0.24571634,  0.18365033, -0.39532098,  1.1073388 ,  0.25882884],
       [-0.0404138 , -1.4639482 , -2.4870509 , -1.13236939, -0.37703079,
         2.26470549,  1.15645665, -0.26645281, -0.36123787, -0.71577669],
       [ 0.88707854, -0.26230725, -0.94072927,  0.18883262,  0.89761512,
         1.60660139, -1.00081086,  0.29518683,  0.30324055,  1.01554217]])

Function

In [31]:
def softmax(x):
  return (np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True))

def scaled_dot_product_attention(q, k, v, mask=None):
  d_k = q.shape[-1]
  scaled = np.matmul(q, k.T) / math.sqrt(d_k)
  if mask is not None:
    scaled = scaled + mask
  attention = softmax(scaled)
  out = np.matmul(attention, v)
  return out, attention

In [32]:
values, attention = scaled_dot_product_attention(q, k, v, mask=mask)
print("Q\n", q)
print("K\n", k)
print("V\n", v)
print("New V\n", values)
print("Attention\n", attention)
     

Q
 [[-0.62216175 -0.87703221 -0.8588049   0.03792949 -0.20157045  1.52719172
   0.06464875 -1.03676197 -0.12062276 -1.21094341]
 [-1.01937969 -0.41035475 -0.19833812 -0.66828901  1.100473   -0.58715359
   0.56277128  0.82087674  1.15482673 -0.26369845]
 [ 0.95904511 -0.51248354  0.38496119 -0.13471253  1.18755381 -1.26213277
  -0.23695832 -0.58007255  0.08457898 -0.67136831]
 [-0.4952777   0.2332489   0.98893229 -0.93413198  1.41187294 -0.01222578
  -0.47614516 -0.50825264 -0.42175741  0.37247078]]
K
 [[-1.01024684  0.84562912  1.30285974 -0.22746246 -0.07658451  0.37897226
  -0.64012633  1.40195369 -0.1959627   0.9703588 ]
 [ 0.15327607 -1.67591322  1.29513144  1.20657875 -0.80070956  0.74721339
   0.46489243  0.19798146  0.68631049 -0.32847494]
 [ 0.92051831  0.1253748  -0.57490041  0.471459    0.08155867 -1.30435789
   1.13935736  0.91164766  0.68222847  1.49772013]
 [ 1.15367826  0.72312258  0.22482272  1.93959019 -0.93613158 -0.26276539
   1.1286182   0.13041126 -1.82527202  0.569